# 06 · Predict from a new questionnaire
The example below is fictional. Edit its values using the allowed inputs from `outputs/reports/input_schema.json`. Provide all keys and use Python `None` or JSON `null` for an unanswered item. The model imputes unanswered items. At least one valid response is required.

The highest predicted class probability supplies the screening class. `probability_high` is the probability assigned to the dataset’s High EPDS screening label. The three probabilities sum to one. This model is for research demonstration.

In [1]:
from pathlib import Path
import sys

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

import json
from src.predict import predict_risk
from src.preprocessing import REPORTS, write_json

questionnaire = {
    "Age": 28,
    "Number of the latest pregnancy": 2,
    "Residence": "City",
    "Education Level": "University",
    "Marital status": "Married",
    "Occupation before latest pregnancy": "Teacher",
    "Monthly income before latest pregnancy": "10000 to 20000",
    "Husband's education level": "College",
    "Disease before pregnancy": "None",
    "History of pregnancy loss": "None",
    "Family type": "Nuclear",
    "Number of household members": "2 to 5",
    "Major changes or losses during pregnancy": "No",
    "Pregnancy length": "9 months",
    "Pregnancy plan": "Yes",
    "Regular checkups": "Yes",
    "Fear of pregnancy": "No",
    "Diseases during pregnancy": "None",
    "Mode of delivery": "Caesarean Section",
    "Depression before pregnancy": "Negative",
    "Depression during pregnancy": "Negative",
}

result = predict_risk(questionnaire)
print(json.dumps(result, indent=2))
write_json(REPORTS / "example_request.json", questionnaire)
write_json(REPORTS / "example_response.json", result)

{
  "model": "SVM",
  "predicted_screening_class": "Low",
  "probability_high": 0.21990771873143097,
  "class_probabilities": {
    "Low": 0.5538002488124466,
    "Medium": 0.22629203245612248,
    "High": 0.21990771873143097
  }
}


## Handoff to the API team
Load `models/best_model.joblib` once at server startup using `joblib.load` with the same Python dependencies. Pass the parsed request dictionary and loaded artifact to `predict_risk(questionnaire, artifact)`. Return the result as JSON. Translate input `ValueError` to a 422 response. Load only a trusted model file. Keep questionnaire payloads out of application logs.